In [ ]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import webbrowser
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import pandas as pd

###########################
# Data Manipulation / Model
###########################

# Read data directly from CSV
from CRUD import CRUD

db = CRUD()

records = db.read()
df = pd.DataFrame(records)


# Create the good dataframe (matching your original)
df_good = df[["age_upon_outcome", "animal_id", "animal_type", "breed", "color", "date_of_birth", "name", "outcome_type",
          "sex_upon_outcome"]]

#AutoGenerates new ID number for new animal records
def generate_new_id():
    records = db.read()
    ids = []
    for record in records:
        animal_id = record.get("animal_id")
        if animal_id and animal_id.startswith("A"):
            try:
                ids.append(int(animal_id[1:]))
            except:
                pass
    if ids:
        return f"A{max(ids)+1}"

    return "A000001"

#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)

# Load logo with your original filename
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#This sets up the Add new animal field for new animals
app.layout = html.Div([
    html.H3("Add New Animal"),
    
    html.Div([
        dcc.Input(
            id='new-name',
            placeholder='Animal Name',
            type='text'
        ),
        dcc.Input(
            id='new-animal-type',
            placeholder='Animal Type',
            type='text'
        ),
        dcc.Input(
            id='new-breed',
            placeholder='Breed',
            type='text'
        ),
        dcc.Input(
            id='new-color',
            placeholder='Color',
            type='text'
        ),
        html.Button(
            'Create Animal',
            id='create-btn',
            n_clicks=0
        )
    ]),
    
    html.Div(id='create-status'),
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='hidden-div', style={'display': 'none'}),
                 html.A([
                     html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                              height=100, width=100
                              )
                 ],
                     href='https://www.snhu.edu', target='_blank'),
                 html.Center(html.B(html.H1(children=[
                     'Kameron Hinman-Mitchell',
                     html.Br(),
                     'SNHU - CS-340',
                 ])))
             ]
             ),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div(
        
        dcc.RadioItems(
            id='filter_type',
            options=[
                {'label': "All (Reset)", 'value': "All"},
                {'label': "Water Rescue", 'value': "WaterRes"},
                {'label': "Mountain or Wilderness Rescue", 'value': "MountainRes"},
                {'label': "Disaster or Individual Tracking Rescue", 'value': "DisasterRes"},
            ],
            value="All"
        )
    ),
    html.Hr(),


    
    html.Hr(),
    
    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df_good.columns],
                         data=df_good.to_dict('records'),
                        editable=False,             # disable editing of data
                        filter_action="native",     # Enable filtering
                        sort_action="native",       # Enable sorting
                        column_selectable=False,    # Disable column selecting
                        row_selectable="single",    # Allow one row to be selected at a time
                        row_deletable=False,        # Disable deletions
                        selected_columns=[],        # No default column selected
                        selected_rows=[0],          # The first row is selected by default
                        page_action="native",       # Enable pages for table
                        page_current=0,             # Sets current page to the first one
                        page_size=10                # Only 10 table rows per page
                        ),
    html.Br(),
    html.Hr(),
    html.Div(
        dcc.Dropdown(
            id='graph_type',
            style=dict(
                width='40%',
                verticalAlign="middle"
            ),
            options=[
                {'label': "Bar Graph", 'value': 0},
                {'label': "Pie Chart", 'value': 1}
            ],
            value=0
        )
    ),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id', 'data'),
    [
        Input('filter_type', 'value'),
        Input('create-btn', 'n_clicks')
    ]
)
def update_dashboard(filter_type, n_clicks):

    records = db.read()
    filtered_df = pd.DataFrame(records)

    if filter_type == 'WaterRes':

        filtered_df = filtered_df[
            (filtered_df['animal_type'] == 'Dog') &
            (filtered_df['breed'].isin([
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ])) &
            (filtered_df['sex_upon_outcome'] == 'Intact Male') &
            (filtered_df['age_upon_outcome_in_weeks'] >= 26.0) &
            (filtered_df['age_upon_outcome_in_weeks'] <= 156.0)
        ]

    elif filter_type == 'MountainRes':

        filtered_df = filtered_df[
            (filtered_df['animal_type'] == 'Dog') &
            (filtered_df['breed'].isin([
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog"
            ])) &
            (filtered_df['sex_upon_outcome'] == 'Intact Male') &
            (filtered_df['age_upon_outcome_in_weeks'] >= 26.0) &
            (filtered_df['age_upon_outcome_in_weeks'] <= 156.0)
        ]

    elif filter_type == 'DisasterRes':

        filtered_df = filtered_df[
            (filtered_df['animal_type'] == 'Dog') &
            (filtered_df['breed'].isin([
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ])) &
            (filtered_df['sex_upon_outcome'] == 'Intact Female') &
            (filtered_df['age_upon_outcome_in_weeks'] >= 20.0) &
            (filtered_df['age_upon_outcome_in_weeks'] <= 300.0)
        ]

    filtered_df = filtered_df[df_good.columns]

    return filtered_df.to_dict('records')
# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
    Input('graph_type', 'value')])
def update_graphs(viewData, graphData):
    # Check if we have data
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available", style={'textAlign': 'center', 'padding': '50px'})
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if breed column exists
    if 'breed' not in dff.columns or len(dff) == 0:
        return html.Div("No breed data available", style={'textAlign': 'center', 'padding': '50px'})
    
    if graphData == 0:  # Bar Graph
        fig = px.histogram(
            dff,
            x='breed',
            height=750,
            width=1000,
            color='breed',
            title='Breed distribution',
            text_auto=True,
        ).update_xaxes(categoryorder='total descending')
        
        return dcc.Graph(figure=fig)
    
    elif graphData == 1:  # Pie Chart
        fig = px.pie(
            dff,
            names='breed',
            title='Breed Distribution',
            height=500,
            width=1000
        )
        fig.update_traces(textposition='inside')
        fig.update_layout(uniformtext_minsize=12, uniformtext_mode='hide')
        return dcc.Graph(figure=fig)

# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]
    
# This callback will update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Check if we have data
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available", style={'textAlign': 'center', 'padding': '50px'})
    
    # Check if a row is selected
    if index is None or len(index) == 0:
        return html.Div("Select an animal to see location", style={'textAlign': 'center', 'padding': '50px'})
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if index is valid
    if index[0] >= len(dff):
        return html.Div("Invalid selection", style={'textAlign': 'center', 'padding': '50px'})
    
    row = index[0]
    
    # Austin TX is at [30.75,-97.48]
    # Check if we have location columns in the original data
    if 'location_lat' in df.columns and 'location_long' in df.columns:
        # Get the original animal_id to find coordinates
        animal_id = dff.iloc[row]['animal_id']
        original_record = df[df['animal_id'] == animal_id]
        
        if len(original_record) > 0:
            lat = original_record.iloc[0]['location_lat']
            lon = original_record.iloc[0]['location_long']
            
            # Handle possible NaN values
            if pd.isna(lat) or pd.isna(lon):
                lat, lon = 30.75, -97.48
        else:
            lat, lon = 30.75, -97.48
    else:
        # Use default coordinates if location data not available
        lat, lon = 30.75, -97.48
    
    # Get animal info for popup
    animal_name = dff.iloc[row].get('name', 'Unknown')
    animal_breed = dff.iloc[row].get('breed', 'Unknown')
    
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(animal_breed),
                dl.Popup([
                    html.H4("Animal Name"),
                    html.P(animal_name)
                ])
            ])
        ])
    ]
@app.callback(
    Output('create-status', 'children'),
    Input('create-btn', 'n_clicks'),
    State('new-name', 'value'),
    State('new-animal-type', 'value'),
    State('new-breed', 'value'),
    State('new-color', 'value')
)
def create_animal(
        n_clicks,
        name,
        animal_type,
        breed,
        color):

    if n_clicks == 0:
        return ""

    animal = {

        "animal_id": generate_new_id(),

        "name": name,

        "animal_type": animal_type,

        "breed": breed,

        "color": color,

        "age_upon_outcome": "",

        "date_of_birth": "",

        "outcome_type": "",

        "sex_upon_outcome": ""
    }

    result = db.create_single(animal)

    if result:

        return f"Animal created successfully."

    return "Insert failed."

# Run app
if __name__ == '__main__':
    url = "http://127.0.0.1:8051"
    app.run(debug=True, port=8051)